# Transform step: cleaning the Online Retail II dataset

This notebook justifies every transformation applied to the dataset before loading it into SQL, then Power BI.
The functions live in `transform.py`. This notebook applies them one by one.

Each step follows the same structure:
1. **Finding**: what the data shows
2. **Decision**: what I do and why
3. **Application**: function call
4. **Measurement**: how much data is affected
5. **Check**: test that the result is the expected one

Vocabulary note: a row of the dataset is an **invoice line** (one product in one invoice). An invoice (`Invoice`) can contain several lines.

In [ ]:
import pandas as pd
import transform as t
import extract as e

pd.set_option("display.max_columns", None)

c:\Users\kamil\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading

In [ ]:
df = e.extract()
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [4]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

### Measurement tool

At each step, we record the number of lines before and after and the impact on revenue (Price × Quantity).

In [5]:
log = []


def apply_step(df, function):
    lines_before = len(df)
    revenue_before = (df["Price"] * df["Quantity"]).sum()
    df = function(df)
    revenue_after = (df["Price"] * df["Quantity"]).sum()
    log.append({
        "step": function.__name__,
        "lines before": lines_before,
        "lines after": len(df),
        "lines removed": lines_before - len(df),
        "% lines removed": round((lines_before - len(df)) / lines_before * 100, 2),
        "revenue change (£)": round(float(revenue_after - revenue_before), 2), })
    print(log[-1])
    return df

## 1. Column types

### Finding

In [6]:
print(df["Customer ID"].dtype)
print(df["InvoiceDate"].dtype)
df[["Customer ID", "InvoiceDate"]].head()

float64
str


,Customer ID,InvoiceDate
0,13085.0,2009-12-01 07:45:00
1,13085.0,2009-12-01 07:45:00
2,13085.0,2009-12-01 07:45:00
3,13085.0,2009-12-01 07:45:00
4,13085.0,2009-12-01 07:45:00


### Decision
`Customer ID` is a float, while it is an integer identifier. I convert it to `Int64`, the pandas integer type that accepts missing values (a regular `int` would fail because of the NaN).
`InvoiceDate` is text: I convert it to a date.

In [7]:
df = apply_step(df, t.fix_types)

{'step': 'fix_types', 'lines before': 1067371, 'lines after': 1067371, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


### Check

In [8]:
print(df["Customer ID"].dtype == "Int64")
print(pd.api.types.is_datetime64_any_dtype(df["InvoiceDate"]))

True
True


## 2. Text normalization

### Finding
Values with extra spaces or lowercase letters:

In [9]:
for column in ["StockCode", "Description", "Invoice"]:
    values = df[column].dropna()
    cleaned = values.str.strip().str.upper()
    print(column, (values != cleaned).sum())

StockCode 3472
Description 218940
Invoice 0


### Decision
I remove leading and trailing spaces and convert everything to uppercase on `StockCode` and `Description`. <br> `Invoice` is already clean, so there is nothing to fix.

In [10]:
df = apply_step(df, t.normalize_text)

{'step': 'normalize_text', 'lines before': 1067371, 'lines after': 1067371, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


## 3. Country standardization

### Finding

In [11]:
df["Country"].value_counts().to_frame().T

Country,United Kingdom,EIRE,Germany,France,Netherlands,Spain,Switzerland,Belgium,Portugal,Australia,Channel Islands,Italy,Norway,Sweden,Cyprus,Finland,Austria,Denmark,Unspecified,Greece,Japan,USA,Poland,United Arab Emirates,Israel,Hong Kong,Singapore,Malta,Iceland,Canada,Lithuania,RSA,Bahrain,Brazil,Thailand,Korea,European Community,Lebanon,West Indies,Bermuda,Nigeria,Czech Republic,Saudi Arabia
count,981330,17866,17624,14330,5140,3811,3189,3123,2620,1913,1664,1534,1455,1364,1176,1049,938,817,756,663,582,535,535,500,371,364,346,299,253,228,189,169,126,94,76,63,61,58,54,34,32,30,10


### Decision
Goal: a consistent naming format for countries.
- **EIRE** → Ireland (Éire is the Irish name of Ireland)
- **RSA** → Republic of South Africa
- **West Indies** → Caribbean
- **USA** → United States of America
- **Korea** → South Korea. The name is ambiguous (North or South?). Assumption based on the commercial context of the dataset.
- **European Community** → missing value: old, obsolete term that does not refer to a country.
- **Unspecified** → missing value.

In [12]:
df = apply_step(df, t.standardize_countries)

{'step': 'standardize_countries', 'lines before': 1067371, 'lines after': 1067371, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


### Measurement

In [13]:
missing = df["Country"].isna().sum()
percent = round(missing/len(df)*100, 3)
print(f"{missing} missing countries out of {len(df)} lines, i.e {percent} %")

817 missing countries out of 1067371 lines, i.e 0.077 %


## 4. Duplicates

### Finding

In [14]:
n_duplicates = df.duplicated().sum()
print(f"{n_duplicates} duplicates, i.e. {n_duplicates / len(df) * 100:.1f} % of lines")

34335 duplicates, i.e. 3.2 % of lines


A duplicate is a line identical to another one on every column. <br>
Example of duplicates :

In [15]:
df[df.duplicated(keep=False)].sort_values(["Invoice", "StockCode"]).head(9)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom


### Decision
These are exact duplicates: same invoice, same product, same quantity, same price, etc. <br>
Assumption: transactions recorded twice by mistake. I remove them.

In [16]:
df = apply_step(df, t.remove_duplicates)

{'step': 'remove_duplicates', 'lines before': 1067371, 'lines after': 1033036, 'lines removed': 34335, '% lines removed': 3.22, 'revenue change (£)': -431716.87}


### Check

In [17]:
print("Duplicates left:", df.duplicated().sum())

Duplicates left: 0


## 5. Missing descriptions

### Finding

In [18]:
no_description = df[df["Description"].isna()]
print(f"{len(no_description)} lines without description")
print(no_description["Price"].value_counts())

4275 lines without description
Price
0.0    4275
Name: count, dtype: int64


### Decision
When a description is missing, the price is always 0 (checked above). These lines are not sales, so I remove them.

In [19]:
df = apply_step(df, t.remove_missing_descriptions)

{'step': 'remove_missing_descriptions', 'lines before': 1033036, 'lines after': 1028761, 'lines removed': 4275, '% lines removed': 0.41, 'revenue change (£)': 0.0}


### Check

In [20]:
no_description = df[df["Description"].isna()]
print(f"{len(no_description)} lines without description")

0 lines without description


## 6. Lines that are not products

### Finding
Lines containing the non-product words:

In [21]:
print(t.NON_PRODUCT_WORDS)
non_products = df[df["Description"].str.contains(t.NON_PRODUCT_WORDS, case=False)]
print(non_products["Description"].value_counts().to_string())

postage|discount|dotcom|manual|test|samples|debt|bank charges|amazon|commission|adjustment
Description
POSTAGE                                2079
DOTCOM POSTAGE                         1423
MANUAL                                 1392
DISCOUNT                                173
SAMPLES                                 102
BANK CHARGES                            100
ADJUSTMENT BY JOHN ON 26/01/2010 16      38
AMAZON FEE                               36
DOTCOMGIFTSHOP GIFT VOUCHER £20.00       26
ADJUSTMENT BY JOHN ON 26/01/2010 17      26
DOTCOMGIFTSHOP GIFT VOUCHER £30.00       24
SOLD AS SET ON DOTCOM                    20
AMAZON                                   18
ADJUSTMENT                               18
THIS IS A TEST PRODUCT.                  16
CRUK COMMISSION                          16
DOTCOM                                   15
DOTCOMGIFTSHOP GIFT VOUCHER £10.00       14
MUG , DOTCOMGIFTSHOP.COM                 10
DOTCOMGIFTSHOP GIFT VOUCHER £50.00        6
ADJUST BAD DEBT  

### Decision
POSTAGE, DOTCOM POSTAGE, DISCOUNT, ADJUSTMENT, BANK CHARGES, AMAZON FEE, TEST... reflect the company's internal logic (fees, adjustments, tests), not product sales. I remove them.
I made sure that the list above does not contain real sales.

In [22]:
df = apply_step(df, t.remove_non_products)

{'step': 'remove_non_products', 'lines before': 1028761, 'lines after': 1023141, 'lines removed': 5620, '% lines removed': 0.55, 'revenue change (£)': 84876.41}


### Check

In [23]:
non_products = df[df["Description"].str.contains(t.NON_PRODUCT_WORDS, case=False)]
print(f"Non-products lines left : {len(non_products)}")

Non-products lines left : 0


## 7. Order errors

### Finding
Zero-price lines:

In [24]:
zero_price = df[df["Price"] == 0]
print(f"{len(zero_price)} zero-price lines")
print(f"of which {zero_price['Invoice'].str.startswith('C').sum()} are refunds")
print(f"of which {(zero_price['Quantity'] < 0).sum()} have a negative quantity")

1609 zero-price lines
of which 0 are refunds
of which 696 have a negative quantity


Part of the zero-price lines describe orders that went wrong.

In [25]:
errors = zero_price[zero_price["Description"].str.contains(t.ERROR_WORDS, case=False)]
print(f"{len(errors)} order error lines in the zero-prices lines")
print(f"i.e. {round(len(errors)/len(zero_price)*100, 2)} % of errors lines in the zero lines")
print(errors["Description"].value_counts().to_string())

371 order error lines in the zero-prices lines
i.e. 23.06 % of errors lines in the zero lines
Description
DAMAGED                            99
DAMAGES                            88
MISSING                            30
SMASHED                             9
THROWN AWAY                         9
UNSALEABLE, DESTROYED.              9
DAMAGES?                            7
CRUSHED                             7
WET DAMAGED                         5
NO STOCK                            3
PRINTING SMUDGES/THROWN AWAY        3
WET PALLET                          3
LOST                                2
WET                                 2
DISCOLOURED                         2
DAMAGES, LOST BITS ETC              2
LOST IN SPACE                       2
MISSING?                            2
MOULDY                              2
WET/RUSTY                           2
WATER DAMAGED                       2
?MISSING                            2
WRONGLY CODED 20713                 2
DAMAGES WAX         

**Are all order errors at price 0?**
To check it, I look at the lines that contain an error word **but have a price**:

In [26]:
real_products = df[(df["Price"] != 0) & df["Description"].str.contains(t.ERROR_WORDS, case=False)]
print(real_products["Description"].unique().tolist())

['MIXED NUTS LIGHT GREEN BOWL']


The only lines with an error word and a price are "MIXED NUTS LIGHT GREEN BOWL": a real product, sold at a normal price. It is matched only because its name contains "mixed".
**Conclusion: every real order error has a price of 0.**

### Decision
I **keep** the order errors. These lines are not sales, but they record real operational problems: damaged, missing, mixed-up or badly made products.
Keeping them makes it possible to analyse these problems in the future.
They do not distort revenue since their price is 0. Deleting them will be considered as a lost of information. 

I group them into 5 categories:
- **QUALITY_ERROR**: bad quality, damp, rust, dirt, mould, incorrectly made...
- **MIXING_ERROR**: mixed-up products
- **DAMAGE_ERROR**: damaged, smashed, wet products...
- **MISSING_ERROR**: missing or lost products
- **SOLD_ERROR**: barcode, stock or sales errors

**Detection rule: price = 0 and an error word in the description.**
Since every order error has a price of 0 (shown above), the price condition does not lose any error. It protects real products whose name contains an error word, like "MIXED NUTS LIGHT GREEN BOWL".
It also lets me keep simple word lists: for example "mixed" alone for MIXING_ERROR, which also catches errors such as "85123A MIXED" or "21733 MIXED".

In [27]:
df = apply_step(df, t.categorize_errors)

{'step': 'categorize_errors', 'lines before': 1023141, 'lines after': 1023141, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


### Measurement

In [28]:
error_lines = df[df["Description"].isin(t.ERROR_CATEGORIES.keys())]
error_lines["Description"].value_counts()

Description
DAMAGE_ERROR     254
MISSING_ERROR     49
QUALITY_ERROR     40
SOLD_ERROR        23
MIXING_ERROR       5
Name: count, dtype: int64

### Check

In [29]:
error_lines = df[df["Description"].isin(t.ERROR_CATEGORIES.keys())]
print(error_lines["Price"].value_counts())
print("MIXED NUTS kept as a product:", (df["Description"] == "MIXED NUTS LIGHT GREEN BOWL").any())

Price
0.0    371
Name: count, dtype: int64
MIXED NUTS kept as a product: True


## 8. Zero-price lines

### Finding
Zero-price lines that are not order errors:

In [30]:
rest = df[(df["Price"] == 0) & ~df["Description"].isin(t.ERROR_CATEGORIES.keys())]
print(f"{len(rest)} lines")
print(rest["Description"].value_counts().head(30).to_string())

1238 lines
Description
CHECK                                 163
?                                      90
FOUND                                  38
OWL DOORSTOP                           14
POLYESTER FILLER PAD 45X45CM           12
POLYESTER FILLER PAD 40X40CM           10
MAILOUT                                 9
FRENCH BLUE METAL DOOR SIGN 1           9
PICNIC BASKET WICKER LARGE              8
CHECKED                                 8
GIVEN AWAY                              8
BOX OF 24 COCKTAIL PARASOLS             8
RED KITCHEN SCALES                      8
IVORY KITCHEN SCALES                    8
RECIPE BOX PANTRY YELLOW DESIGN         8
FRENCH BLUE METAL DOOR SIGN 8           8
WATERING CAN BLUE ELEPHANT              7
CHILDS GARDEN SPADE BLUE                7
ENAMEL WASH BOWL CREAM                  7
AIRLINE BAG VINTAGE WORLD CHAMPION      7
MINT KITCHEN SCALES                     7
RED RETROSPOT CHARLOTTE BAG             7
FRENCH BLUE METAL DOOR SIGN 4           7
FRENCH BLUE

### Decision
Order errors were handled in the previous step. The remaining zero-price lines are not real sales. I remove them.

In [31]:
df = apply_step(df, t.remove_zero_prices)

{'step': 'remove_zero_prices', 'lines before': 1023141, 'lines after': 1021903, 'lines removed': 1238, '% lines removed': 0.12, 'revenue change (£)': 0.0}


### Check

In [32]:
left = df[(df["Price"] == 0) & ~df["Description"].isin(t.ERROR_CATEGORIES.keys())]
print("Zero-price lines left outside errors:", len(left))

Zero-price lines left outside errors: 0


## 9. Line type

### Decision
I add a `line_type` column to easily tell lines apart in future analyses : 
- **error**
- **sale**
- **refund**: refund (invoice starting with C)

In [33]:
df = apply_step(df, t.classify_lines)

{'step': 'classify_lines', 'lines before': 1021903, 'lines after': 1021903, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


### Measurement

In [34]:
df["line_type"].value_counts()

line_type
sale      1003610
refund      17922
error         371
Name: count, dtype: int64

### Check

In [35]:
print("Lines without line_type:", df["line_type"].isna().sum())
print("Error lines with a price:", len(df[(df["line_type"] == "error") & (df["Price"] != 0)]))
print("Zero-price lines outside errors:", len(df[(df["Price"] == 0) & (df["line_type"] != "error")]))

Lines without line_type: 0
Error lines with a price: 0
Zero-price lines outside errors: 0


## 10. One description per product

### Finding
Some StockCodes have several different descriptions:

In [36]:
products = df[df["line_type"] != "error"]
groups = products.groupby("StockCode")
n_descriptions = groups["Description"].nunique()
print(f"{(n_descriptions > 1).sum()} StockCodes with several descriptions")
n_descriptions.sort_values(ascending=False).head(10)

599 StockCodes with several descriptions


StockCode
22346     4
20685     4
22345     4
22344     4
22384     4
23196     4
23236     4
22191     3
17107D    3
22853     3
Name: Description, dtype: int64

These differences come from two cases: small typing variations (an extra comma, a double space...) or a description that was reworded over time while still referring to the same product. <br>
Example :

In [37]:
print(df[(df["StockCode"] == "23196") & (df["Invoice"] == "557858")][["StockCode", "Description"]])
print(df[(df["StockCode"] == "23196") & (df["Invoice"] == "581217")][["StockCode", "Description"]])

       StockCode                    Description
762883     23196  RETRO LEAVES MAGNETIC NOTEPAD
        StockCode                    Description
1061269     23196  VINTAGE LEAF MAGNETIC NOTEPAD


### Decision

I give each StockCode its most frequent description, to get a single description per product. This keeps the data consistent and makes later processing easier. <br> Error lines keep their category.

In [38]:
n_errors_before = (df["line_type"] == "error").sum()
df = apply_step(df, t.unify_descriptions)

{'step': 'unify_descriptions', 'lines before': 1021903, 'lines after': 1021903, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


### Check

In [39]:
n_descriptions = df[df["line_type"] != "error"].groupby("StockCode")["Description"].nunique()
print("StockCodes with several descriptions:", (n_descriptions > 1).sum())

# no error category became a product description
n_errors_after = df["Description"].isin(t.ERROR_CATEGORIES.keys()).sum()
print("Error lines before:", n_errors_before, "/ After:", n_errors_after)

StockCodes with several descriptions: 0
Error lines before: 371 / After: 371


## 11. Outliers
An outlier is a value that differs significantly from the rest of the data.

Invoices identified as inconsistent:

In [40]:
outliers = df[df["Invoice"].isin(t.OUTLIER_INVOICES)]
outliers[["Invoice", "InvoiceDate", "StockCode", "Description", "Quantity", "Price", "Customer ID"]]

,Invoice,InvoiceDate,StockCode,Description,Quantity,Price,Customer ID
587080,541431,2011-01-18 10:01:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346
587085,C541433,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346
748132,556444,2011-06-10 15:28:00,22502,PICNIC BASKET WICKER SMALL,60,649.50,15098
748153,C556448,2011-06-10 15:39:00,22502,PICNIC BASKET WICKER SMALL,-60,4.95,15098
1065882,581483,2011-12-09 09:15:00,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446
1065883,C581484,2011-12-09 09:27:00,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446


### Decision
I remove each abnormal sale and its refund:
- **581483 / C581484 and 541431 / C541433**: huge quantities. Their impact on net revenue is zero, but they distort gross revenue, average quantities and product rankings.
- **556444 / C556448**: 60 baskets at about £600 each. Completley inconsistent. <br>
I remove these outliers because they distort product-level results: a single abnormal sale can make a product look far more successful than it really is.

In [41]:
df = apply_step(df, t.remove_outliers)

{'step': 'remove_outliers', 'lines before': 1021903, 'lines after': 1021897, 'lines removed': 6, '% lines removed': 0.0, 'revenue change (£)': -38673.0}


### Check

In [42]:
print("Outlier invoices left:", df["Invoice"].isin(t.OUTLIER_INVOICES).sum())

Outlier invoices left: 0


## 12. Unknown customers

### Finding

In [43]:
no_customer = df["Customer ID"].isna().sum()
print(f"{no_customer} lines without Customer ID, i.e. {no_customer / len(df) * 100:.1f} %")

227467 lines without Customer ID, i.e. 22.3 %


In [44]:
df[df["Customer ID"].isna()]["line_type"].value_counts()

line_type
sale      226767
error        371
refund       329
Name: count, dtype: int64

### Decision
I **keep** these lines: they are real sales. Removing them would lose a large share of revenue.
I replace the missing identifier with **-1** (unknown customer), so that every line referring to an unknown customer will be easier to filter in future analyses.

In [45]:
df = apply_step(df, t.fill_unknown_customers)

{'step': 'fill_unknown_customers', 'lines before': 1021897, 'lines after': 1021897, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


### Check

In [46]:
print("Lines without Customer ID:", df["Customer ID"].isna().sum())
print("Customer -1 lines:", (df["Customer ID"] == -1).sum(), "/ expected:", no_customer)

Lines without Customer ID: 0
Customer -1 lines: 227467 / expected: 227467


## 13. Revenue column

### Decision
`Revenue = Price × Quantity`. Refunds have a negative quantity, hence a negative Revenue: the sum directly gives net revenue.

In [47]:
df = apply_step(df, t.add_revenue)

{'step': 'add_revenue', 'lines before': 1021897, 'lines after': 1021897, 'lines removed': 0, '% lines removed': 0.0, 'revenue change (£)': 0.0}


### Check

In [48]:
print(f"Net revenue: {df['Revenue'].sum():.2f} £")
df.groupby("line_type")["Revenue"].sum().round(2)

Net revenue: 18901737.11 £


line_type
error            0.00
refund     -470802.37
sale      19372539.48
Name: Revenue, dtype: float64

## Summary

Impact of each step:

In [49]:
summary = pd.DataFrame(log)
summary

,step,lines before,lines after,lines removed,% lines removed,revenue change (£)
0,fix_types,1067371,1067371,0,0.00,0.00
1,normalize_text,1067371,1067371,0,0.00,0.00
2,standardize_countries,1067371,1067371,0,0.00,0.00
3,remove_duplicates,1067371,1033036,34335,3.22,-431716.87
4,remove_missing_descriptions,1033036,1028761,4275,0.41,0.00
5,remove_non_products,1028761,1023141,5620,0.55,84876.41
6,categorize_errors,1023141,1023141,0,0.00,0.00
7,remove_zero_prices,1023141,1021903,1238,0.12,0.00
8,classify_lines,1021903,1021903,0,0.00,0.00
9,unify_descriptions,1021903,1021903,0,0.00,0.00


In [50]:
removed = summary["lines removed"].sum()
start = summary["lines before"].iloc[0]
print(f"Lines at start: {start}")
print(f"Lines at end  : {len(df)}")
print(f"Removed       : {removed} ({removed / start * 100:.1f} %)")

Lines at start: 1067371
Lines at end  : 1021897
Removed       : 45474 (4.3 %)


In [51]:
df.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                object
line_type              object
Revenue               float64
dtype: object

In [52]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_type,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,sale,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,sale,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,sale,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,sale,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,sale,30.0
